In [ ]:
import sys
import torch

import numpy as np
import trimesh
import plotly.graph_objects as go

In [ ]:
sys.path.append("..")

In [ ]:
from utils.grasp_utils import get_handmodel
from model.hand_opt import AdamGraspTransfer

## Info

In [57]:
source_gripper = "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

## Hand Models

In [ ]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [ ]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

## Source Gripper Pose

In [ ]:
grasp_pose = torch.zeros(9)
grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3] = 1
grasp_pose[7] = 1 
print("Pose:", grasp_pose)


grasp_dofs_lower = source_model.dynamic_joints_q_lower.squeeze(0).clone()
grasp_dofs_mid = torch.tensor(source_model.dynamic_joints_q_mid)

grasp_dofs = grasp_dofs_lower + grasp_dofs_mid
# grasp_dofs = grasp_dofs_mid # for shadowhand
# scale = 0.5
# grasp_dofs = scale * torch.rand_like(grasp_dofs_mid) * grasp_dofs_mid + grasp_dofs_lower



print("Dofs:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

In [ ]:
sample_grasp_q.shape

## Grasp Transfer Optimization

In [ ]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [ ]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

In [ ]:
print(q_traj.shape)
best_q = q_traj[21, -1]
print(best_q.shape)

In [ ]:
best_q.shape[0]

In [ ]:
target_model.dynamic_joints_q_lower.shape

In [ ]:
midjoints = torch.tensor(target_model.dynamic_joints_q_mid)
print(midjoints)

In [ ]:
target_model.dynamic_joints_q_upper[0]

In [ ]:
target_model.dynamic_joints_q_lower[0]

In [ ]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, midjoints), dim=0)

In [ ]:
print(best_q)

## Visualize results

In [ ]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
vis_data += target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("../logs/viz_gtransfer_test.html")


In [ ]:
base_pose = torch.zeros(9)
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
base_pose[3] = 1
base_pose[7] = 1 
print("Base Pose:", base_pose)

base_q = torch.cat((base_pose, midjoints), dim=0)
print("Base Q:", base_q)